# Setup

### Imports

In [1]:
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import cloudpickle

### Constants

In [2]:
SEED = 42
FEATURES = ["temperature", "humidity", "wind_direction", "wind_speed", "precipitation", "light"]

### Set splitting

In [3]:
def split_set(X, y, seed):
    X_tune, X_test, y_tune, y_test = train_test_split(X, y, test_size=.2, random_state=seed)
    X_train, X_validate, y_train, y_validate = train_test_split(X_tune, y_tune, test_size=.2, random_state=seed)
    return ((X_train, y_train), (X_validate, y_validate), (X_test, y_test))

In [4]:
def get_sets(data, seed=42, do_split=True, drop_y_nan=True):
    y_labels = [label for label in data.columns if label.startswith("future_")]

    if drop_y_nan:
        data = data.dropna(subset=y_labels)
    X = data.drop([*y_labels], axis="columns")
    y = data[y_labels]

    if do_split:
        return split_set(X, y, seed)
    else:
        return (X, y)

### Metrics

In [5]:
from sklearn.metrics import r2_score, root_mean_squared_error

def compute_metrics(estimator, sets):
    n_features = sets[0][2].shape[1]
    metrics = [("r2", r2_score), ("RMSE", root_mean_squared_error)]
    metric_outputs = {"feature": [*(sets[0][2].iloc[:, i].name for i in range(n_features)), "all"]}
    predictions = [estimator.predict(set[1]) for set in sets]

    for metric_label, metric in metrics:
        for set_idx, (set_label, _, y) in enumerate(sets):
            for feature_idx in range(n_features):
                metric_outputs.setdefault(f"{metric_label}_{set_label}", [])
                metric_outputs[f"{metric_label}_{set_label}"].append(metric(y.iloc[:, feature_idx], [row[feature_idx] for row in predictions[set_idx]]))
            metric_outputs[f"{metric_label}_{set_label}"].append(metric(y, predictions[set_idx]))
    print(pd.DataFrame(metric_outputs))

def fit_and_score(model, sets):
    model.fit(*(sets[0]))
    compute_metrics(model, [("train", *(sets[0])), ("validate", *(sets[1]))])


### Plotting

In [6]:
def plot_predictions(model, input, hours, just: list[str] | None = None):
    labels = input.columns.to_list()
    labels.remove("time")


    rows = []
    for row in input.iloc:
        rows.extend([row] * hours)
    inputs = pd.DataFrame(rows, columns=input.columns)
    inputs["prediction_offset"] = list(range(hours)) * len(input)
    predictions = model.predict(inputs)

    if just is not None:
        allowed_outputs = [labels.index(output) for output in just]
        old_predictions = predictions
        predictions = []
        for row in old_predictions:
            prediction_row = []
            for i, prediction in enumerate(row):
                if i in allowed_outputs:
                    prediction_row.append(prediction)
            predictions.append(prediction_row)
        labels = just

    plt.figure()
    plt.plot(range(hours), predictions)
    plt.legend(labels)
    plt.show()

### Pipeline functions

In [7]:
def drop_columns(X, columns):
    return X.drop(columns, axis="columns", errors="ignore")

def drop_nan(X):
    return X.dropna()

def move_column(X, idx, col_name):
    column_to_move = X.pop(col_name)
    X.insert(idx, col_name, column_to_move)
    return X

def rename_columns(X, name_map):
    return X.rename(columns=name_map)

def drop_before(X, before_date):
    return X.drop(X[X["time"] < before_date.timestamp()].index)

def drop_after(X, after_date):
    return X.drop(X[X["time"] >= after_date.timestamp()].index)

def _encode_cyclic(value, range_start, range_stop):
    value = (value - range_start) / (range_stop - range_start)
    x = np.sin(2 * np.pi * value)
    y = np.cos(2 * np.pi * value)
    return x, y

def encode_cyclic(X, feature, range_start, range_stop, new_feature_suffix=None, remove_old=False, get_feature=None):
    if get_feature is None:
        def get_feature(row):
            return row[feature]
    if new_feature_suffix is None:
        new_feature_suffix = feature
    X[f"{new_feature_suffix}_sin"] = X[[feature]].apply(lambda row: _encode_cyclic(get_feature(row), range_start, range_stop)[0], axis="columns")
    X[f"{new_feature_suffix}_cos"] = X[[feature]].apply(lambda row: _encode_cyclic(get_feature(row), range_start, range_stop)[1], axis="columns")
    if remove_old:
        X.drop([feature], inplace=True)
    return X

def split_time(X, year=True, month=True, day=True, hour=True, minute=False, second=False, make_cyclic=False, remove_old=True):
    X = X.copy()
    if year:
        X["year"] = X[["time"]].apply(lambda row: datetime.fromtimestamp(row["time"]).year, axis="columns")
    if month:
        if not make_cyclic:
            X["month"] = X[["time"]].apply(lambda row: datetime.fromtimestamp(row["time"]).month, axis="columns")
        else:
            encode_cyclic(X, "time", 1, 12, new_feature_suffix="month", get_feature=lambda row: datetime.fromtimestamp(row["time"]).month)
    if day:
        if not make_cyclic:
            X["day"] = X[["time"]].apply(lambda row: datetime.fromtimestamp(row["time"]).day, axis="columns")
        else:
            encode_cyclic(X, "time", 1, 31, new_feature_suffix="day", get_feature=lambda row: datetime.fromtimestamp(row["time"]).day)
    if hour:
        if not make_cyclic:
            X["hour"] = X[["time"]].apply(lambda row: datetime.fromtimestamp(row["time"]).hour, axis="columns")
        else:
            encode_cyclic(X, "time", 0, 23, new_feature_suffix="hour", get_feature=lambda row: datetime.fromtimestamp(row["time"]).hour)
    if minute:
        if not make_cyclic:
            X["minute"] = X[["time"]].apply(lambda row: datetime.fromtimestamp(row["time"]).minute, axis="columns")
        else:
            encode_cyclic(X, "time", 0, 59, new_feature_suffix="minute", get_feature=lambda row: datetime.fromtimestamp(row["time"]).minute)
    if second:
        if not make_cyclic:
            X["second"] = X[["time"]].apply(lambda row: datetime.fromtimestamp(row["time"]).second, axis="columns")
        else:
            encode_cyclic(X, "time", 0, 59, new_feature_suffix="second", get_feature=lambda row: datetime.fromtimestamp(row["time"]).second)


    if remove_old:
        X.drop(["time"], axis="columns", inplace=True)

    return X

### Read data

In [8]:
raw_data = pd.read_parquet(Path("../data/historical_data_06102.parquet"))
raw_data_limited = raw_data.dropna().sample(n=100000, random_state=SEED)
sets_limited = get_sets(raw_data_limited, seed=SEED)
raw_data_ultra_limited = drop_before(raw_data.dropna(), datetime.now() - timedelta(days=365*2)).sample(n=10000, random_state=SEED)
sets_ultra_limited = get_sets(raw_data_ultra_limited, seed=SEED)

In [9]:
raw_data_iot = pd.read_parquet(Path("../data/IoT_data.parquet"))
sets_iot = get_sets(raw_data_iot.dropna(), seed=SEED)

# Experiments

## Setup Sets

In [10]:
preprocessor_base = Pipeline([
    ("NaN_dropper", FunctionTransformer(drop_nan)),
    ("time_spliter", FunctionTransformer(split_time)),
])

In [11]:
preprocessor_scaled = Pipeline([
    ("base", preprocessor_base),
    ("scaler", StandardScaler())
])

In [12]:
preprocessor_cyclic = Pipeline([
    ("NaN_dropper", FunctionTransformer(drop_nan)),
    ("time_spliter", FunctionTransformer(lambda X: split_time(X, make_cyclic=True))),
])

In [13]:
preprocessor_cyclic_scaled = Pipeline([
    ("base", preprocessor_cyclic),
    ("scaler", StandardScaler())
])

In [14]:
preprocessor_no_year = Pipeline([
    ("NaN_dropper", FunctionTransformer(drop_nan)),
    ("time_spliter", FunctionTransformer(lambda X: split_time(X, year=False))),
])

In [15]:
preprocessor_cyclic_no_year = Pipeline([
    ("NaN_dropper", FunctionTransformer(drop_nan)),
    ("time_spliter", FunctionTransformer(lambda X: split_time(X, year=False, make_cyclic=True))),
])

## Best model

In [16]:
best_model = MultiOutputRegressor(
    HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_features=0.6,
        max_iter=200,
        max_leaf_nodes=None,
        min_samples_leaf=33,
        random_state=42,
    )
)

In [ ]:
prepreprocessor = Pipeline(
    [
        ("NaN_dropper", FunctionTransformer(lambda X: X.dropna())),
    ]
)
preprocessor = Pipeline(
    [
        ("prepreprocessor", prepreprocessor),
        (
            "time_spliter",
            FunctionTransformer(
                lambda X: split_time(
                    X, year=False, make_cyclic=True, remove_old=True
                )
            ),
        ),
    ]
)
model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

new_data = prepreprocessor.fit_transform(raw_data)
new_data = new_data.sample(n=1000000)

sets = get_sets(new_data, seed=SEED)
fit_and_score(model, sets)
fitted_best_model = model

In [ ]:
plot_predictions(model, sets_limited[1][0][:1], 24 * 7, just=["temperature"])

## Save model

In [ ]:
with open("../model.pkl", "wb") as f:
    cloudpickle.dump(best_model, f, protocol=5)

## Base

In [ ]:
model = MultiOutputRegressor(HistGradientBoostingRegressor(max_depth=5, random_state=SEED))
preprocessor = Pipeline([
    ("NaN_dropper", FunctionTransformer(drop_nan)),
    ("time_spliter", FunctionTransformer(split_time)),
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])
fit_and_score(model, sets_limited)

plot_predictions(model, sets_limited[1][0][:1], 24 * 7)
plot_predictions(model, sets_limited[1][0][:1], 24 * 7, just=["temperature"])

## Models

### HistGradientBoostringRegressor

In [ ]:
model = MultiOutputRegressor(HistGradientBoostingRegressor(random_state=SEED))
model = Pipeline([
    ("preprocessor", preprocessor_base),
    ("model", model)
])
params = {
    "model__estimator__max_depth": range(5, 15),
    "model__estimator__learning_rate": [0.05, 0.1, 0.2,],
    "model__estimator__min_samples_leaf": range(30, 50),
    "model__estimator__max_iter": [10, 25, 50, 100, 200],
    "model__estimator__max_features": [0.2, 0.4, 0.6, 0.8, 1.0],
    "model__estimator__max_leaf_nodes": [None, *range(21, 60, 5)]
}
search = RandomizedSearchCV(model, params, random_state=SEED)

search_sets = sets_limited
search.fit(*search_sets[0])

print(search.best_estimator_)
compute_metrics(search.best_estimator_, [("train", *(search_sets[0])), ("validate", *(search_sets[1]))])

In [19]:
model = MultiOutputRegressor(HistGradientBoostingRegressor(random_state=SEED))
model = Pipeline([
    ("preprocessor", preprocessor_cyclic_no_year),
    ("model", model)
])
params = {
    "model__estimator__max_depth": range(5, 15),
    "model__estimator__learning_rate": [0.05, 0.1, 0.2,],
    "model__estimator__min_samples_leaf": range(30, 50),
    "model__estimator__max_iter": [10, 25, 50, 100, 200],
    "model__estimator__max_features": [0.2, 0.4, 0.6, 0.8, 1.0],
    "model__estimator__max_leaf_nodes": [None, *range(21, 60, 5)]
}
search = RandomizedSearchCV(model, params, random_state=SEED)

search_sets = sets_iot
search.fit(*search_sets[0])

print(search.best_estimator_)
compute_metrics(search.best_estimator_, [("train", *(search_sets[0])), ("validate", *(search_sets[1]))])

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('NaN_dropper',
                                  FunctionTransformer(func=<function drop_nan at 0x799eadcba5c0>)),
                                 ('time_spliter',
                                  FunctionTransformer(func=<function <lambda> at 0x799eadb1bec0>))])),
                ('model',
                 MultiOutputRegressor(estimator=HistGradientBoostingRegressor(learning_rate=0.05,
                                                                              max_depth=8,
                                                                              max_features=0.6,
                                                                              max_iter=200,
                                                                              max_leaf_nodes=None,
                                                                              min_samples_leaf=33,
                                                                

In [18]:
model = MultiOutputRegressor(HistGradientBoostingRegressor(max_depth=10, random_state=SEED))
model = Pipeline([
    ("preprocessor", preprocessor_base),
    ("model", model)
])
fit_and_score(model, sets_iot)

                 feature  r2_train  r2_validate    RMSE_train  RMSE_validate
0     future_temperature  0.820362     0.644339      2.038732       2.711510
1        future_humidity  0.871417     0.721723      7.308531      10.307547
2  future_wind_direction  0.560738     0.336299     50.078789      62.575365
3      future_wind_speed  0.876425     0.739130      0.773323       1.166638
4   future_precipitation  0.430799    -0.102809      0.141291       0.190775
5           future_light  0.763628     0.552586  10095.164570   13209.280405
6                    all  0.720562     0.481878   1692.584206    2214.372040


In [ ]:
model = MultiOutputRegressor(HistGradientBoostingRegressor(max_depth=10, random_state=SEED))
model = Pipeline([
    ("preprocessor", preprocessor_no_year),
    ("model", model)
])
fit_and_score(model, sets_limited)

In [ ]:
model = MultiOutputRegressor(HistGradientBoostingRegressor(max_depth=10, random_state=SEED))
model = Pipeline([
    ("preprocessor", preprocessor_cyclic),
    ("model", model)
])
fit_and_score(model, sets_limited)

## Scaled

In [ ]:
model = MultiOutputRegressor(HistGradientBoostingRegressor(max_depth=10, random_state=SEED))
model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", model)
])
fit_and_score(model, sets_limited)

## Random forest

In [ ]:
model = MultiOutputRegressor(RandomForestRegressor(max_depth=5, random_state=SEED))
model = Pipeline([
    ("preprocessor", preprocessor_base),
    ("model", model)
])
fit_and_score(model, sets_limited)

## SVM

In [ ]:
model = MultiOutputRegressor(SVR(kernel="rbf"))
model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", model)
])
fit_and_score(model, sets_ultra_limited)

In [ ]:
model = MultiOutputRegressor(SVR(kernel="rbf", C=100, gamma=0.1, epsilon=0.1))
model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", model)
])
fit_and_score(model, sets_ultra_limited)

In [ ]:
model = MultiOutputRegressor(SVR(kernel="rbf", C=100, gamma=0.1, epsilon=0.1))
model = Pipeline([
    ("preprocessor", preprocessor_cyclic_scaled),
    ("model", model)
])
fit_and_score(model, sets_ultra_limited)

In [ ]:
model = MultiOutputRegressor(SVR(kernel="poly", degree=2))
model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", model)
])
fit_and_score(model, sets_ultra_limited)

In [ ]:
model = MultiOutputRegressor(SVR(kernel="poly", degree=3))
model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", model)
])
fit_and_score(model, sets_ultra_limited)

## Untrained dates

In [ ]:
for include_year in [True, False]:
    pipeline_old_dates = Pipeline([
        ("before_date_dropper", FunctionTransformer(lambda X: drop_before(X, datetime(2025, 1, 1) - timedelta(days=365*2)))),
        ("after_date_dropper", FunctionTransformer(lambda X: drop_after(X, datetime(2025, 1, 1)))),
        ("NaN_dropper", FunctionTransformer(drop_nan)),
    ])

    pipeline_new_dates = Pipeline([
        ("before_date_dropper", FunctionTransformer(lambda X: drop_before(X, datetime(2025, 1, 1)))),
        ("NaN_dropper", FunctionTransformer(drop_nan)),
    ])

    print(f"===== OLD {"with" if include_year else "without"} year =====")
    preprocessor = Pipeline([
        ("NaN_dropper", FunctionTransformer(drop_nan)),
        ("time_spliter", FunctionTransformer(lambda X: split_time(X, year=include_year, make_cyclic=True))),
    ])
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", best_model)
    ])
    fit_and_score(model, get_sets(pipeline_old_dates.fit_transform(raw_data_limited), seed=SEED))

    print(f"===== NEW {"with" if include_year else "without"} year =====")
    compute_metrics(model, get_sets(pipeline_new_dates.fit_transform(raw_data_limited), seed=SEED))

## Cyclic wind direction

In [ ]:
preprocessor = Pipeline([
    ("NaN_dropper", FunctionTransformer(drop_nan)),
    ("time_spliter", FunctionTransformer(lambda X: split_time(X, year=False, make_cyclic=True))),
    ("wind_dir_cyclic", FunctionTransformer(lambda X: encode_cyclic(X, "wind_direction", 0, 360, remove_old=False))),
    ("wind_dir_dropper", FunctionTransformer(lambda X: drop_columns(X, ["wind_direction"])))
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

fit_and_score(model, sets_limited)

## Negative rain

In [ ]:
X = sets_limited[0][0]
X[X["precipitation"] < 0]

In [ ]:
predictions = pd.DataFrame(fitted_best_model.predict(sets_limited[1][0]), columns=sets_limited[1][1].columns)
predictions[predictions["future_precipitation"] < 0]

In [ ]:
def add_is_raining(X):
    X["is_raining"] = (X["precipitation"] > 0).astype(int)
    return X

model = Pipeline([
    ("preprocessor", preprocessor_cyclic),
    ("is_raining", FunctionTransformer(add_is_raining)),
    ("model", best_model),
])

fit_and_score(model, sets_limited)

predictions = pd.DataFrame(model.predict(sets_limited[1][0]), columns=sets_limited[1][1].columns)
predictions[predictions["future_precipitation"] < 0]